Reprezentacja tensorów 3D, 4D, 5D

In [1]:
import torch
import tensorflow as tf

# --- 3D: NLP / Audio -> [Batch, SeqLen, Features] ---
# PyTorch
nlp_pt = torch.randn(32, 50, 768)

# TensorFlow
nlp_tf = tf.random.normal([32, 50, 768])
print(f"3D NLP -> PyTorch: {nlp_pt.shape} | TensorFlow: {nlp_tf.shape}")

# --- 4D: Computer Vision ---
# PyTorch (domyślnie NCHW): [Batch, Channels, Height, Width]

cv_pt = torch.randn(16, 3, 224, 224)
# TensorFlow (domyślnie NHWC): [Batch, Height, Width, Channels]

cv_tf = tf.random.normal([16, 224, 224, 3])
print(f"4D CV -> PyTorch (NCHW): {cv_pt.shape} | TensorFlow (NHWC): {cv_tf.shape}")

# --- 5D: Video / Medical Imaging ---
# PyTorch: [Batch, Channels, Frames, Height, Width]
video_pt = torch.randn(4, 3, 16, 112, 112)

# TensorFlow: [Batch, Frames, Height, Width, Channels]
video_tf = tf.random.normal([4, 16, 112, 112, 3])
print(f"5D Video -> PyTorch: {video_pt.shape} | TensorFlow: {video_tf.shape}")

3D NLP -> PyTorch: torch.Size([32, 50, 768]) | TensorFlow: (32, 50, 768)
4D CV -> PyTorch (NCHW): torch.Size([16, 3, 224, 224]) | TensorFlow (NHWC): (16, 224, 224, 3)
5D Video -> PyTorch: torch.Size([4, 3, 16, 112, 112]) | TensorFlow: (4, 16, 112, 112, 3)


Transpozycja wieloosiowa i konwersja NCHW <=> NHWC

In [2]:
import torch
import tensorflow as tf

# --- PYTORCH ---
pt_nchw = torch.randn(8, 3, 64, 64)

# NCHW -> NHWC (zamiana osi: 0->0, 2->1, 3->2, 1->3)
pt_nhwc = pt_nchw.permute(0, 2, 3, 1)
pt_restored = pt_nhwc.permute(0, 3, 1, 2)
print(f"PyTorch NCHW -> NHWC: {pt_nhwc.shape}")

# --- TENSORFLOW ---
tf_nhwc = tf.random.normal([8, 64, 64, 3])

# NHWC -> NCHW (zamiana osi: 0->0, 3->1, 1->2, 2->3)
tf_nchw = tf.transpose(tf_nhwc, perm=[0, 3, 1, 2])
tf_restored = tf.transpose(tf_nchw, perm=[0, 2, 3, 1])
print(f"TensorFlow NHWC -> NCHW: {tf_nchw.shape}")

PyTorch NCHW -> NHWC: torch.Size([8, 64, 64, 3])
TensorFlow NHWC -> NCHW: (8, 3, 64, 64)


Zaawansowany broadcasting, maskowanie i gather

In [3]:
import torch
import tensorflow as tf

# === A. BROADCASTING (Odejmowanie sredniej kanalow) ===
print("=" * 20, "A. BROADCASTING", "=" * 20)

# PyTorch
pt_imgs = torch.ones(4, 3, 32, 32)
pt_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
pt_norm = pt_imgs - pt_mean

# Wyswietlamy wartosci dla pierwszego piksela w kazdym z 3 kanalow (powinno byc 1 - mean)
print(f"PyTorch normalised shape: {pt_norm.shape}")
print(f"PyTorch wartosci dla piksela [0, :, 0, 0]:\n{pt_norm[0, :, 0, 0]}\n")

# TensorFlow
tf_imgs = tf.ones([4, 32, 32, 3])
tf_mean = tf.reshape(tf.constant([0.485, 0.456, 0.406]), [1, 1, 1, 3])
tf_norm = tf_imgs - tf_mean
print(f"TensorFlow normalised shape: {tf_norm.shape}")
print(f"TensorFlow wartosci dla piksela [0, 0, 0, :]:\n{tf_norm[0, 0, 0, :].numpy()}\n")

# === B. MASKOWANIE LOGICZNE (Wyzerowanie ujemnych) ===
print("=" * 20, "B. MASKOWANIE LOGICZNE", "=" * 20)

# PyTorch
pt_data = torch.tensor([[-1.0, 2.0], [3.0, -4.0]])
pt_data[pt_data < 0] = 0.0
print(f"PyTorch po wyzerowaniu ujemnych:\n{pt_data}\n")

# TensorFlow
tf_data = tf.constant([[-1.0, 2.0], [3.0, -4.0]])
tf_data = tf.where(tf_data < 0, 0.0, tf_data)
print(f"TensorFlow po wyzerowaniu ujemnych:\n{tf_data.numpy()}\n")

# === C. GATHER (Wyciaganie elementow po indeksach) ===
print("=" * 20, "C. GATHER", "=" * 20)

# PyTorch
pt_logits = torch.tensor([[0.1, 0.8, 0.1], [0.7, 0.2, 0.1]])
pt_targets = torch.tensor([[1], [0]]) # Wyciagamy klase 1 z 1. wiersza oraz klase 0 z 2. wiersza
pt_selected = torch.gather(pt_logits, dim=1, index=pt_targets)
print("PyTorch wybrane logity (gather):")
print(f"Ksztalt: {pt_selected.shape}")
print(f"Wartosci:\n{pt_selected}\n")

# TensorFlow
tf_logits = tf.constant([[0.1, 0.8, 0.1], [0.7, 0.2, 0.1]])
tf_targets = tf.constant([1, 0])
tf_selected = tf.gather(tf_logits, indices=tf_targets, batch_dims=1)
print("TensorFlow wybrane logity (gather):")
print(f"Ksztalt: {tf_selected.shape}")
print(f"Wartosci:\n{tf_selected.numpy()}\n")

==================== A. BROADCASTING ====================
PyTorch normalised shape: torch.Size([4, 3, 32, 32])
PyTorch wartosci dla piksela [0, :, 0, 0]:
tensor([0.5150, 0.5440, 0.5940])

TensorFlow normalised shape: (4, 32, 32, 3)
TensorFlow wartosci dla piksela [0, 0, 0, :]:
[0.515 0.544 0.594]

==================== B. MASKOWANIE LOGICZNE ====================
PyTorch po wyzerowaniu ujemnych:
tensor([[0., 2.],
        [3., 0.]])

TensorFlow po wyzerowaniu ujemnych:
[[0. 2.]
 [3. 0.]]

==================== C. GATHER ====================
PyTorch wybrane logity (gather):
Ksztalt: torch.Size([2, 1])
Wartosci:
tensor([[0.8000],
        [0.7000]])

TensorFlow wybrane logity (gather):
Ksztalt: (2,)
Wartosci:
[0.8 0.7]



Notacja Einsteina

In [4]:
import torch
import tensorflow as tf

# A. Mnozenie macierzy (A @ B)
pt_A, pt_B = torch.randn(2, 3), torch.randn(3, 4)
tf_A, tf_B = tf.random.normal([2, 3]), tf.random.normal([3, 4])
pt_C = torch.einsum('ik,kj->ij', pt_A, pt_B)
tf_C = tf.einsum('ik,kj->ij', tf_A, tf_B)

# B. Batch Attention Scores: [Batch, Heads, SeqLen, Dim] x [Batch, Heads, SeqLen, Dim]
pt_Q = torch.randn(2, 4, 5, 8)
pt_K = torch.randn(2, 4, 6, 8)
tf_Q = tf.random.normal([2, 4, 5, 8])
tf_K = tf.random.normal([2, 4, 6, 8])
pt_attn = torch.einsum('bhid,bhjd->bhij', pt_Q, pt_K)
tf_attn = tf.einsum('bhid,bhjd->bhij', tf_Q, tf_K)
print(f"Einsum Attention -> PyTorch: {pt_attn.shape} | TensorFlow: {tf_attn.shape}")

Einsum Attention -> PyTorch: torch.Size([2, 4, 5, 6]) | TensorFlow: (2, 4, 5, 6)


Łączenie i rozdzielanie tensorów

In [5]:
import torch
import tensorflow as tf

print("=" * 20, "1. PIERWOTNE TENSORY WEJŚCIOWE", "=" * 20)

# --- PYTORCH ---
pt_t1, pt_t2 = torch.ones(2, 3), torch.zeros(2, 3)
print("PyTorch - pt_t1 (sama jedynki, kształt [2, 3]):")
print(f"{pt_t1}\n")
print("PyTorch - pt_t2 (same zera, kształt [2, 3]):")
print(f"{pt_t2}\n")

# --- TENSORFLOW ---
tf_t1, tf_t2 = tf.ones([2, 3]), tf.zeros([2, 3])
print("TensorFlow - tf_t1 (jedynki):")
print(f"{tf_t1.numpy()}\n")
print("TensorFlow - tf_t2 (zera):")
print(f"{tf_t2.numpy()}\n")
print("=" * 20, "2. OPERACJE ŁĄCZENIA I ROZDZIELANIA", "=" * 20)

# PyTorch - operacje
pt_cat = torch.cat([pt_t1, pt_t2], dim=0) # [4, 3]
pt_stack = torch.stack([pt_t1, pt_t2], dim=0) # [2, 2, 3]
pt_splits = torch.split(pt_cat, split_size_or_sections=2, dim=0) # [2x [2, 3]]

# TensorFlow - operacje
tf_cat = tf.concat([tf_t1, tf_t2], axis=0) # [4, 3]
tf_stack = tf.stack([tf_t1, tf_t2], axis=0) # [2, 2, 3]
tf_splits = tf.split(tf_cat, num_or_size_splits=2, axis=0) # [2x [2, 3]]
print(f"Cat/Concat shape -> PT: {pt_cat.shape} | TF: {tf_cat.shape}")
print(f"Stack shape -> PT: {pt_stack.shape} | TF: {tf_stack.shape}")
print(f"Liczba części po split -> PT: {len(pt_splits)} (kształt: {pt_splits[0].shape}) | TF: {len(tf_splits)} (kształt: {tf_splits[0].shape})\n")
print("Wynik PyTorch cat (połączenie wzdłuż dim=0):\n", pt_cat)
print("\nWynik PyTorch stack (utworzenie nowego wymiaru dim=0):\n", pt_stack)

==================== 1. PIERWOTNE TENSORY WEJŚCIOWE ====================
PyTorch - pt_t1 (sama jedynki, kształt [2, 3]):
tensor([[1., 1., 1.],
        [1., 1., 1.]])

PyTorch - pt_t2 (same zera, kształt [2, 3]):
tensor([[0., 0., 0.],
        [0., 0., 0.]])

TensorFlow - tf_t1 (jedynki):
[[1. 1. 1.]
 [1. 1. 1.]]

TensorFlow - tf_t2 (zera):
[[0. 0. 0.]
 [0. 0. 0.]]

==================== 2. OPERACJE ŁĄCZENIA I ROZDZIELANIA ====================
Cat/Concat shape -> PT: torch.Size([4, 3]) | TF: (4, 3)
Stack shape -> PT: torch.Size([2, 2, 3]) | TF: (2, 2, 3)
Liczba części po split -> PT: 2 (kształt: torch.Size([2, 3])) | TF: 2 (kształt: (2, 3))

Wynik PyTorch cat (połączenie wzdłuż dim=0):
 tensor([[1., 1., 1.],
        [1., 1., 1.],
        [0., 0., 0.],
        [0., 0., 0.]])

Wynik PyTorch stack (utworzenie nowego wymiaru dim=0):
 tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[0., 0., 0.],
         [0., 0., 0.]]])
